# Practice 26 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — Basics

### Task 1: penguins — measurement profile

Load `sns.load_dataset('penguins')`. Drop all rows with nulls.

Build a view where **rows = measurements, columns = species, values = mean**.

Three questions — no hints:

- Get the mean of each numeric column per species. Reshape it so species are columns and measurements are rows. Think carefully about which level to unstack and why.
- Use `.xs()` to extract all species' mean `bill_length_mm`.
- Which `(species, measurement)` pair has the highest mean value? Use NumPy — no `.idxmax()`.

In [19]:
# Your code here
penguins = sns.load_dataset('penguins').dropna()
mcol = penguins.select_dtypes(include='number').columns
mcol
penguinsm = penguins.groupby('species')[mcol].mean()
#penguinsm.T
penguinsm.stack().unstack('species')

penguinsm.unstack().xs('bill_length_mm')

pp = penguinsm.unstack()
pp.index[np.argmax(pp)]

('body_mass_g', 'Gentoo')

---
## Level 2 — Transformations

### Task 2: weather — build your own pipeline

A weather dataset across 5 cities and 6 months is provided.

Design and build a `.pipe()` chain — **write all functions yourself, no scaffold**. Requirements:
- Exactly 3 functions
- One must use `transform` to add a city-level metric (e.g. city mean or city rank)
- One must classify rows using `.apply()`
- One must add a flag based on two conditions

After the pipeline:
- Stack the numeric columns so each row is `(city, month, attribute)`. Store as `weather_long`.
- Use `.xs()` to extract all months and attributes for `'Oslo'`.
- Use `.xs()` to extract the `'temp_c'` attribute across all cities and months.

In [38]:
# Starter data — don't change this
np.random.seed(7)
cities = ['Oslo', 'Cairo', 'Sydney', 'Toronto', 'Mumbai']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
weather = pd.DataFrame({
    'city':         np.repeat(cities, len(months)),
    'month':        months * len(cities),
    'temp_c':       np.round(np.random.uniform(-10, 40, size=30), 1),
    'rainfall_mm':  np.round(np.random.uniform(0, 200, size=30), 1),
    'humidity_pct': np.random.randint(30, 95, size=30),
})

# Your code here
def city_temp_mean(df):
    df['city_temp_mean'] = df.groupby('city')['temp_c'].transform('mean')
    return df
def cat_humidity(df):
    df['humidity_cat'] = df.apply(lambda row: 'High' if row['rainfall_mm']>100 and row['humidity_pct']>80
                                  else 'Mid' if row['rainfall_mm']>100 or row['humidity_pct']>80
                                  else 'Low', axis =1)
    return df
def flag_bad(df):
    df['bad'] = (df['temp_c']<10) & (df['humidity_pct']>80)
    return df 


res = (
    weather.copy()
    .pipe(city_temp_mean)
    .pipe(cat_humidity)
    .pipe(flag_bad)
)



#n_cols = weather.select_dtypes(include='number').columns
#n_cols
weather
weather_long = weather.set_index(['city','month']).stack()

oslo = weather_long.xs('Oslo', level='city')
oslo
temp_c = weather_long.xs('temp_c', level = 2)
temp_c


city     month
Oslo     Jan      -6.2
         Feb      29.0
         Mar      11.9
         Apr      26.2
         May      38.9
         Jun      16.9
Cairo    Jan      15.1
         Feb      -6.4
         Mar       3.4
         Apr      15.0
         May      24.0
         Jun      30.2
Sydney   Jan       9.0
         Feb      -6.7
         Mar       4.4
         Apr      35.5
         May       0.7
         Jun      12.6
Toronto  Jan      36.6
         Feb      -8.8
         Mar      20.0
         Apr      37.5
         May       1.5
         Jun      17.4
Mumbai   Jan      35.5
         Feb      -3.3
         Mar      16.2
         Apr      27.5
         May      23.5
         Jun      13.4
dtype: float64

---
## Level 3 — Aggregation

### Task 3: diamonds — which cut stands out?

Load `sns.load_dataset('diamonds')`.

**Q1.** For each numeric column, compute this ratio using NumPy:

> variance of the group means (by `cut`) / mean of the within-group variances

A high ratio means the groups are far apart relative to how spread out each group is — a good separator. Which numeric column best separates diamond cuts?

**Q2.** Among `'Ideal'` cut diamonds, what fraction have a `price` above 2× the overall median price? Use NumPy — no filtering with groupby.

In [53]:
# Your code here

diamonds = sns.load_dataset('diamonds')
nc = diamonds.select_dtypes(include='number').columns
nc_ratio = {}
for col in nc:
    gm = diamonds.groupby('cut', observed=True)[col].mean()
    gv = diamonds.groupby('cut', observed=True)[col].var()
    nc_ratio[col] = np.var(gm)/np.mean(gv)
max(nc_ratio, key = nc_ratio.get)

ideal = diamonds[diamonds['cut'] == 'Ideal'].copy()
ideal['high_price'] = ideal['price']>2*np.median(ideal['price'])
np.mean(ideal['high_price'])


np.float64(0.3069927149552225)

---
## Level 4 — Real-world

### Task 4: tips — full analysis

Load `sns.load_dataset('tips')`. No scaffolding.

1. Add `tip_pct = tip / total_bill`. Set a `(day, time)` MultiIndex. Sort it.
2. Use `.xs()` to retrieve all **Sunday Dinner** rows.
3. Build a `.pipe()` chain with 2 functions of your choosing. At least one must use `transform`. End with a summary of something non-obvious.
4. Do smokers tip a higher percentage than non-smokers? Answer using NumPy boolean indexing — not groupby.
5. Create a reshaped view: mean `tip_pct` with `day` as rows and `time` as columns. Use `.xs()` to extract one row or column.

In [ ]:
# Your code here

tips = sns.load_dataset('tips')
tips['tip_pct'] = tips['tip']/tips['total_bill']
tips = tips.set_index(['day','time']).sort_index()
tips.xs('Sun',level='day').xs('Dinner')
# why is this wrong?
#tips.xs('Sun',level='day').xs('Dinner', level = 'time')

def big_meal(df):
    df['big_meal'] = df['total_bill']>50
    return df 

def time_avg_tip(df):
    df['time_avg_tip'] = df.groupby(level = 'time', observed = True)['tip_pct'].transform('mean')
    return df
tips_res = (
    tips.copy()
    .pipe(big_meal)
    .pipe(time_avg_tip)
)

tips_smoker = tips[tips['smoker']=='Yes']['tip_pct']
tips_nsmoker = tips[tips['smoker']=='No']['tip_pct']
np.mean(tips_smoker)>np.mean(tips_nsmoker)


reshape = tips.groupby(level=['day','time'], observed=True)['tip_pct'].mean().unstack()

reshape.xs('Thur')

day
Thur    0.159744
Fri     0.158916
Sat     0.153152
Sun     0.166897
Name: Dinner, dtype: float64